In [ ]:
from __future__ import annotations

import hashlib
import re
import time
from dataclasses import dataclass
from pathlib import Path

from pymongo import UpdateOne
from pypdf import PdfReader

from app.utils.helpers import utc_now
from app.utils.logging import configure_logging, log_event

logger = configure_logging()


@dataclass
class ExtractedPage:
    page_number: int
    text: str


@dataclass
class TextChunk:
    chunk_id: str
    material_id: str
    project_id: str
    user_id: str
    page_number: int
    chunk_index: int
    text: str
    source_file_name: str | None = None


class DocumentService:

    def __init__(
        self,
        database,
        chunk_size: int = 1200,
        chunk_overlap: int = 200,
    ):
        if chunk_size <= 0:
            raise ValueError(
                "chunk_size must be greater than zero."
            )

        if chunk_overlap < 0:
            raise ValueError(
                "chunk_overlap cannot be negative."
            )

        if chunk_overlap >= chunk_size:
            raise ValueError(
                "chunk_overlap must be smaller than chunk_size."
            )

        self.database = database
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    # --------------------------------------------------------
    # PDF
    # --------------------------------------------------------

    def extract_pdf(
        self,
        file_path: str | Path,
    ) -> list[ExtractedPage]:

        path = Path(file_path)

        if not path.exists():
            raise FileNotFoundError(
                f"PDF file not found: {path}"
            )

        if path.suffix.lower() != ".pdf":
            raise ValueError(
                "Only PDF files are currently supported."
            )

        reader = PdfReader(str(path))

        pages = []

        for index, page in enumerate(reader.pages):
            text = page.extract_text() or ""

            pages.append(
                ExtractedPage(
                    page_number=index + 1,
                    text=self.clean_text(text),
                )
            )

        return pages

    # --------------------------------------------------------
    # CLEANING
    # --------------------------------------------------------

    @staticmethod
    def clean_text(text: str) -> str:

        if not text:
            return ""

        text = text.replace("\r\n", "\n")
        text = text.replace("\r", "\n")

        text = re.sub(
            r"[ \t]+",
            " ",
            text,
        )

        text = re.sub(
            r"\n{3,}",
            "\n\n",
            text,
        )

        text = re.sub(
            r" *\n *",
            "\n",
            text,
        )

        return text.strip()

    # --------------------------------------------------------
    # CHUNKING
    # --------------------------------------------------------

    def chunk_page(
        self,
        text: str,
        page_number: int,
        material_id: str,
        project_id: str,
        user_id: str,
        source_file_name: str | None = None,
        starting_index: int = 0,
    ) -> list[TextChunk]:

        text = self.clean_text(text)

        if not text:
            return []

        chunks = []

        start = 0
        index = starting_index

        while start < len(text):

            end = min(
                start + self.chunk_size,
                len(text),
            )

            chunk_text = text[start:end].strip()

            if chunk_text:

                chunk_id = self._make_chunk_id(
                    material_id,
                    page_number,
                    index,
                    chunk_text,
                )

                chunks.append(
                    TextChunk(
                        chunk_id=chunk_id,
                        material_id=material_id,
                        project_id=project_id,
                        user_id=user_id,
                        page_number=page_number,
                        chunk_index=index,
                        text=chunk_text,
                        source_file_name=source_file_name,
                    )
                )

                index += 1

            if end >= len(text):
                break

            start = end - self.chunk_overlap

        return chunks

    def chunk_pages(
        self,
        pages: list[ExtractedPage],
        material_id: str,
        project_id: str,
        user_id: str,
        source_file_name: str | None = None,
    ) -> list[TextChunk]:

        chunks = []
        global_index = 0

        for page in pages:

            page_chunks = self.chunk_page(
                text=page.text,
                page_number=page.page_number,
                material_id=material_id,
                project_id=project_id,
                user_id=user_id,
                source_file_name=source_file_name,
                starting_index=global_index,
            )

            chunks.extend(page_chunks)
            global_index += len(page_chunks)

        return chunks

    # --------------------------------------------------------
    # EMBEDDING
    # --------------------------------------------------------

    def embed_chunks(
        self,
        chunks: list[TextChunk],
        ai_service,
        embedding_batch_size: int = 64,
        material_id: str | None = None,
    ) -> list[dict]:
        """
        Embed all chunks with batched calls to the embedding model
        instead of one ai_service.embed_text() call per chunk.

        The sentence-transformer model itself batches its forward
        pass, so one call over `embedding_batch_size` chunks is
        far faster than that many individual calls.
        """

        if not chunks:
            return []

        texts = [chunk.text for chunk in chunks]

        started = time.perf_counter()

        embeddings = ai_service.embed_texts(
            texts=texts,
            task_type="RETRIEVAL_DOCUMENT",
            batch_size=embedding_batch_size,
        )

        elapsed = time.perf_counter() - started

        log_event(
            logger,
            "material.embedding.completed",
            material_id=material_id,
            chunk_count=len(chunks),
            batch_size=embedding_batch_size,
            elapsed_seconds=round(elapsed, 2),
        )

        now = utc_now()

        return [
            {
                "id": chunk.chunk_id,
                "chunk_id": chunk.chunk_id,
                "material_id": chunk.material_id,
                "project_id": chunk.project_id,
                "user_id": chunk.user_id,
                "page_number": chunk.page_number,
                "chunk_index": chunk.chunk_index,
                "text": chunk.text,
                "source_file_name": chunk.source_file_name,
                "embedding": embedding,
                "created_at": now,
                "updated_at": now,
            }
            for chunk, embedding in zip(chunks, embeddings)
        ]

    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    def store_chunks(
        self,
        documents: list[dict],
        material_id: str | None = None,
        batch_size: int = 500,
    ) -> int:
        """
        Persist chunks with pymongo bulk_write() instead of one
        update_one(upsert=True) round-trip per chunk.

        Each operation is still an upsert keyed on the
        deterministic chunk id (see _make_chunk_id), so reprocessing
        the same material overwrites the same documents rather than
        creating duplicates.
        """

        if not documents:
            return 0

        collection = self.database.collection(
            "document_chunks"
        )

        started = time.perf_counter()

        for offset in range(0, len(documents), batch_size):

            batch = documents[offset:offset + batch_size]

            operations = [
                UpdateOne(
                    {
                        "id": document["id"],
                        "user_id": document["user_id"],
                    },
                    {
                        "$set": document,
                    },
                    upsert=True,
                )
                for document in batch
            ]

            # ordered=False lets independent upserts in the batch
            # keep going even if one fails, and lets the driver
            # send them without waiting on each other in sequence.
            collection.bulk_write(
                operations,
                ordered=False,
            )

        elapsed = time.perf_counter() - started

        log_event(
            logger,
            "material.storage.completed",
            material_id=material_id,
            chunk_count=len(documents),
            batch_size=batch_size,
            elapsed_seconds=round(elapsed, 2),
        )

        return len(documents)

    # --------------------------------------------------------
    # FULL PIPELINE
    # --------------------------------------------------------

    def process_material(
        self,
        material: dict,
        file_path: str | Path,
        ai_service=None,
    ) -> dict:

        material_id = material["id"]
        project_id = material["project_id"]
        user_id = material["user_id"]

        materials = self.database.collection(
            "materials"
        )

        materials.update_one(
            {
                "id": material_id,
                "user_id": user_id,
            },
            {
                "$set": {
                    "status": "processing",
                    "processing_status": "processing",
                    "processing_started_at": utc_now(),
                    "processing_error": None,
                    "updated_at": utc_now(),
                }
            },
        )

        pipeline_started = time.perf_counter()

        log_event(
            logger,
            "material.processing.started",
            material_id=material_id,
            file_name=material.get("file_name"),
        )

        try:

            extraction_started = time.perf_counter()

            pages = self.extract_pdf(file_path)

            log_event(
                logger,
                "material.extraction.completed",
                material_id=material_id,
                page_count=len(pages),
                elapsed_seconds=round(
                    time.perf_counter() - extraction_started, 2
                ),
            )

            chunks = self.chunk_pages(
                pages=pages,
                material_id=material_id,
                project_id=project_id,
                user_id=user_id,
                source_file_name=material.get(
                    "file_name"
                ),
            )

            log_event(
                logger,
                "material.chunking.completed",
                material_id=material_id,
                chunk_count=len(chunks),
            )

            if ai_service is None:
                raise RuntimeError(
                    "AIService is required for embedding generation."
                )

            if not chunks:
                # A PDF with no extractable text (e.g. scanned
                # images with no OCR) still completes processing;
                # it just has nothing to embed or store.
                documents = []
                stored = 0

            else:

                documents = self.embed_chunks(
                    chunks=chunks,
                    ai_service=ai_service,
                    material_id=material_id,
                )

                stored = self.store_chunks(
                    documents,
                    material_id=material_id,
                )

            materials.update_one(
                {
                    "id": material_id,
                    "user_id": user_id,
                },
                {
                    "$set": {
                        "status": "completed",
                        "processing_status": "completed",
                        "page_count": len(pages),
                        "chunk_count": stored,
                        "processing_completed_at": utc_now(),
                        "processing_error": None,
                        "updated_at": utc_now(),
                    }
                },
            )

            log_event(
                logger,
                "material.processing.completed",
                material_id=material_id,
                page_count=len(pages),
                chunk_count=stored,
                elapsed_seconds=round(
                    time.perf_counter() - pipeline_started, 2
                ),
            )

            return {
                "material_id": material_id,
                "page_count": len(pages),
                "chunk_count": stored,
                "status": "completed",
            }

        except Exception as exc:

            log_event(
                logger,
                "material.processing.failed",
                material_id=material_id,
                error=str(exc),
                elapsed_seconds=round(
                    time.perf_counter() - pipeline_started, 2
                ),
            )

            materials.update_one(
                {
                    "id": material_id,
                    "user_id": user_id,
                },
                {
                    "$set": {
                        "status": "failed",
                        "processing_status": "failed",
                        "processing_error": str(exc),
                        "processing_failed_at": utc_now(),
                        "updated_at": utc_now(),
                    }
                },
            )

            raise

    @staticmethod
    def _make_chunk_id(
        material_id: str,
        page_number: int,
        chunk_index: int,
        text: str,
    ) -> str:

        raw = (
            f"{material_id}:"
            f"{page_number}:"
            f"{chunk_index}:"
            f"{text}"
        )

        digest = hashlib.sha256(
            raw.encode("utf-8")
        ).hexdigest()[:24]

        return f"chunk_{digest}"
